In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver_accounts = spark.table("digital_banking.silver.silver_accounts")

# Create account dimension with enriched attributes
df_dim_account = df_silver_accounts.select(
    # Primary Key
    F.col("account_id").alias("account_key"),
    
    # Account Identification
    F.col("account_id"),
    
    # Foreign Keys
    F.col("customer_id"),
    F.col("branch_id"),
    
    # Account Classification
    F.coalesce(F.col("account_type"), F.lit("Unknown")).alias("account_type"),
    F.coalesce(F.col("account_status"), F.lit("Unknown")).alias("account_status"),
    F.coalesce(F.col("account_tier"), F.lit("Standard")).alias("account_tier"),
    F.coalesce(F.col("currency"), F.lit("INR")).alias("currency"),
    
    # Account Type Category
    F.when(F.col("account_type").isin(["Savings", "Salary"]), "Deposit Account")
     .when(F.col("account_type").isin(["Current", "Business"]), "Transaction Account")
     .when(F.col("account_type").isin(["Fixed Deposit", "Recurring Deposit"]), "Investment Account")
     .when(F.col("account_type").isin(["Loan", "Credit"]), "Credit Account")
     .otherwise("Other").alias("account_category"),
    
    # Interest Rate Information
    F.col("interest_rate"),
    F.when(F.col("interest_rate") > 0, "Interest Bearing")
     .when(F.col("interest_rate") == 0, "Non-Interest Bearing")
     .otherwise("Unknown").alias("interest_bearing_flag"),
    
    # Temporal Attributes
    F.col("opening_date"),
    F.col("closing_date"),
    
    # Derived Status Indicators
    F.when(F.col("account_status") == "Active", True)
     .otherwise(False).alias("is_active"),
    F.when(F.col("closing_date").isNotNull(), True)
     .otherwise(False).alias("is_closed"),
    
    # Account Age Metrics
    F.datediff(F.current_date(), F.col("opening_date")).alias("days_since_opening"),
    F.floor(F.datediff(F.current_date(), F.col("opening_date")) / 365.25).alias("account_age_years"),
    
    # Account Age Group
    F.when(F.datediff(F.current_date(), F.col("opening_date")) < 180, "0-6 months")
     .when(F.datediff(F.current_date(), F.col("opening_date")) < 365, "6-12 months")
     .when(F.datediff(F.current_date(), F.col("opening_date")) < 730, "1-2 years")
     .when(F.datediff(F.current_date(), F.col("opening_date")) < 1825, "2-5 years")
     .otherwise("5+ years").alias("account_age_group"),
    
    # Closing Date Metrics (for closed accounts)
    F.when(F.col("closing_date").isNotNull(), 
           F.datediff(F.current_date(), F.col("closing_date")))
     .otherwise(None).alias("days_since_closing"),
    
    # Account Lifetime (opening to closing or current date)
    F.when(F.col("closing_date").isNotNull(),
           F.datediff(F.col("closing_date"), F.col("opening_date")))
     .otherwise(F.datediff(F.current_date(), F.col("opening_date"))).alias("account_lifetime_days"),
    
    # Audit Columns
    F.col("updated_at").alias("source_updated_at"),
    F.current_timestamp().alias("dimension_created_at"),
    F.current_timestamp().alias("dimension_updated_at")
)

df_dim_account.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("digital_banking.gold.dim_account")

print(f"Total accounts: {df_dim_account.count()}")

# Display 
display(df_dim_account.limit(10))